
# TraficGenius: Previsão de Trânsito - Los Angeles (METR-LA)
## MVP Acadêmico - Fase 0

**Contexto:** Prever a velocidade média do tráfego em rodovias permite um melhor gerenciamento da malha urbana e auxilia em sistemas de navegação preditiva.
**Objetivo:** Prever a velocidade média do trânsito na rede rodoviária do condado de Los Angeles.
**Dataset:** *METR-LA*. Este é um dos datasets mais famosos no mundo acadêmico para previsão de tráfego, contendo dados de velocidade coletados de 207 sensores (loop detectors) distribuídos pelas rodovias de Los Angeles, com medições a cada 5 minutos.

Neste notebook, abordaremos:
1. Ingestão e EDA (Análise Exploratória)
2. Pré-processamento (Agregação Espacial e Lags Temporais)
3. Modelagem Preditiva (Linear Regression, Random Forest, XGBoost)
4. Avaliação e Comparação


### 1. Preparação do Ambiente e Imports

In [1]:

# Instalando pacotes necessários
!pip install xgboost plotly scikit-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

# Configuração visual padrão
sns.set_theme(style="whitegrid")



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



### 2. Carregamento dos Dados (METR-LA)
Vamos baixar o arquivo CSV do Zenodo. Originalmente, o METR-LA é uma matriz com 207 sensores. Para simplificar a modelagem tradicional (sem usar grafos) e criar um modelo preditivo eficiente, vamos agregar a velocidade média da cidade e prever a velocidade global futura com base no histórico recente.


In [3]:
import pandas as pd
import glob
import os

# 1. Definir o caminho da pasta onde estão os arquivos .parquet
folder_path = r'c:\Users\samuelbarroso\Documents\Desenvolvimento\TraficGenius\dataset'

# 2. Listar todos os arquivos parquet que começam com 'train' (ou use '*.parquet' para todos)
files = glob.glob(os.path.join(folder_path, "train-*.parquet"))

# 3. Ler todos os arquivos e unir (concatenar) em um único DataFrame
# Usamos uma 'list comprehension' para ler cada arquivo da lista
df_list = [pd.read_parquet(f) for f in files]
df_final = pd.concat(df_list, ignore_index=True)

# 4. Ordenar pela coluna de data (Start_Time)
# No Parquet, as datas geralmente já vêm no formato correto, sem precisar de parse_dates
df_sorted = df_final.sort_values(by='Start_Time', ascending=False)

# 5. Exibir o resultado
print(f"Total de arquivos unidos: {len(files)}")
print(df_sorted.head())



Total de arquivos unidos: 7
                ID   Source  Severity                     Start_Time  \
3662007  A-3682074  Source1         2            2023-03-31 23:30:00   
3659354  A-3678205  Source1         2            2023-03-31 23:25:30   
3647141  A-3660237  Source1         2            2023-03-31 23:21:00   
3665682  A-3687382  Source1         2            2023-03-31 23:19:00   
3649166  A-3663193  Source1         2  2023-03-31 23:18:00.000000000   

                              End_Time  Start_Lat   Start_Lng    End_Lat  \
3662007            2023-03-31 23:53:00  42.419172  -83.088509  42.431590   
3659354            2023-03-31 23:38:30  34.068219 -117.384067  34.068406   
3647141            2023-03-31 23:38:30  38.361154 -121.970167  38.356127   
3665682            2023-03-31 23:40:00  42.959427  -85.672227  42.957582   
3649166  2023-03-31 23:36:30.000000000  32.909061  -96.732217  32.903161   

            End_Lng  Distance(mi)  ... Roundabout Station   Stop  \
3662007  -83.0

In [2]:

print("Baixando dados METR-LA (Los Angeles)... Isso pode levar alguns segundos.")
url = "https://zenodo.org/records/5146275/files/METR-LA.csv"
df_raw = pd.read_csv(url)

# O CSV tem a primeira coluna como tempo e o resto como sensores
df_raw.rename(columns={df_raw.columns[0]: 'datetime'}, inplace=True)
df_raw['datetime'] = pd.to_datetime(df_raw['datetime'])
df_raw.set_index('datetime', inplace=True)

display(df_raw.head())
print(f"\nDimensões do dataset original: {df_raw.shape}")


Baixando dados METR-LA (Los Angeles)... Isso pode levar alguns segundos.


,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
datetime,,,,,,,,,,,,,,,,,,,,,
2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000,68.750000,65.125,67.125,59.625000,62.750000,...,45.625000,65.500,64.500000,66.428571,66.875,59.375000,69.000000,59.250000,69.000000,61.875
2012-03-01 00:05:00,62.666667,68.555556,65.444444,62.444444,64.444444,68.111111,65.000,65.000,57.444444,63.333333,...,50.666667,69.875,66.666667,58.555556,62.000,61.111111,64.444444,55.888889,68.444444,62.875
2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,65.375000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857143,62.000
2012-03-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000
2012-03-01 00:20:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000



Dimensões do dataset original: (34272, 207)



### 3. Análise Exploratória de Dados (EDA)
Vamos calcular a velocidade média em todos os 207 sensores e visualizar o padrão de trânsito ao longo do tempo.


In [3]:

# Calculando a média global de velocidade na rede
df = pd.DataFrame()
df['mean_speed'] = df_raw.mean(axis=1)

# Amostragem para visualizar (Plotar uma semana de dados)
fig = px.line(df.iloc[:2016], y='mean_speed', 
              title="Velocidade Média do Trânsito em Los Angeles (Primeira Semana)",
              labels={'datetime': 'Data e Hora', 'mean_speed': 'Velocidade Média (mph)'})
fig.show()

# Distribuição
fig2 = px.histogram(df, x='mean_speed', nbins=50, marginal="box",
                   title="Distribuição da Velocidade Média")
fig2.show()



### 4. Pré-processamento e Feature Engineering (Lags Temporais)
Em Séries Temporais, o melhor previsor do futuro é o passado recente.
Vamos criar características (*Features*) de *Lag* (atraso). Por exemplo, prever a velocidade no tempo $T$ usando a velocidade em $T-1, T-2, T-3$. Além disso, vamos extrair hora e dia da semana.


In [4]:

# Criando features temporais
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek

# Criando Lags (Histórico dos últimos 3 períodos = últimos 15 minutos)
# Como a medição é a cada 5 min, lag_1 é 5 min atrás, lag_2 é 10 min atrás, etc.
for i in range(1, 4):
    df[f'speed_lag_{i}'] = df['mean_speed'].shift(i)

# Remover linhas com valores nulos causados pelo shift()
df.dropna(inplace=True)

display(df.head())

# Separação Treino / Teste (Temporal - Prevenindo Data Leakage)
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# Definindo Target
X_train = train_df.drop(columns=['mean_speed'])
y_train = train_df['mean_speed']

X_test = test_df.drop(columns=['mean_speed'])
y_test = test_df['mean_speed']

print(f"\nTamanho do Treino: {len(X_train)} amostras")
print(f"Tamanho do Teste: {len(X_test)} amostras")


,mean_speed,hour,day_of_week,speed_lag_1,speed_lag_2,speed_lag_3
datetime,,,,,,
2012-03-01 00:15:00,0.000000,0,3,61.858149,62.335816,62.305038
2012-03-01 00:20:00,0.000000,0,3,0.000000,61.858149,62.335816
2012-03-01 00:25:00,61.759259,0,3,0.000000,0.000000,61.858149
2012-03-01 00:30:00,61.107057,0,3,61.759259,0.000000,0.000000
2012-03-01 00:35:00,61.046325,0,3,61.107057,61.759259,0.000000



Tamanho do Treino: 27415 amostras
Tamanho do Teste: 6854 amostras



### 5. Modelagem Preditiva
Usaremos 3 modelos para prever a velocidade atual com base no histórico imediato e na hora/dia.


In [5]:

# 1. Regressão Linear (Baseline)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

# 2. Random Forest Regressor
# Limitamos n_estimators para treinar rápido no Colab
rf_model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# 3. XGBoost Regressor
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

print("Modelos treinados com sucesso!")


Modelos treinados com sucesso!



### 6. Avaliação e Comparação
Métricas: MAE (Erro médio absoluto), RMSE (Raiz do Erro Quadrático Médio) e R².


In [6]:

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {'Modelo': name, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

results = []
results.append(evaluate_model('Regressão Linear', y_test, lr_preds))
results.append(evaluate_model('Random Forest', y_test, rf_preds))
results.append(evaluate_model('XGBoost', y_test, xgb_preds))

results_df = pd.DataFrame(results)

display(results_df.style.highlight_min(subset=['MAE', 'RMSE'], color='lightgreen')
                        .highlight_max(subset=['R²'], color='lightgreen'))

fig = px.bar(results_df, x='Modelo', y='R²', title="Comparação de R² entre Modelos",
             color='Modelo', text_auto='.3f')
fig.show()


,Modelo,MAE,RMSE,R²
0,Regressão Linear,1.066404,4.530474,0.933849
1,Random Forest,1.443750,5.404549,0.905861
2,XGBoost,1.623673,5.767912,0.892777



### 7. Análise Visual (Real vs Predito) e Conclusão
Vamos visualizar um pequeno trecho de tempo no conjunto de teste (ex: um dia de trânsito) para ver como o XGBoost acompanhou as quedas de velocidade reais.


In [7]:

# Pegando apenas 288 amostras (24 horas * 12 medições de 5 min)
amostras = 288
test_comparison = pd.DataFrame({
    'Índice (Tempo)': range(amostras),
    'Real (mph)': y_test.values[:amostras],
    'XGBoost Predito (mph)': xgb_preds[:amostras]
})

fig = go.Figure()
fig.add_trace(go.Scatter(x=test_comparison['Índice (Tempo)'], y=test_comparison['Real (mph)'],
                    mode='lines', name='Real (Ground Truth)'))
fig.add_trace(go.Scatter(x=test_comparison['Índice (Tempo)'], y=test_comparison['XGBoost Predito (mph)'],
                    mode='lines', name='Predito (XGBoost)', line=dict(dash='dot')))

fig.update_layout(title="XGBoost: Previsão vs Realidade (Janela de 24 horas)",
                  xaxis_title="Tempo (Intervalos de 5 min)",
                  yaxis_title="Velocidade (mph)")
fig.show()



### 8. Conclusão Científica
A abordagem de utilizar a própria Série Temporal (*Lags* do passado) para prever o futuro no dataset **METR-LA** demonstrou altíssima eficácia.
- Como o trânsito muda de forma contínua, prever os próximos 5 minutos usando os 15 minutos anteriores se beneficia muito de modelos não-lineares (*XGBoost*), superando a Regressão Linear que não lida tão bem com picos abruptos de lentidão.
